In [ ]:
import os
import gc
import torch
import torchaudio
import pandas as pd
import evaluate
from dataclasses import dataclass
from typing import Any
from datasets import Dataset
from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)
from peft import LoraConfig, get_peft_model

# Clear memory
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


# CONFIGURATION - > Local PC Paths

PROJECT_ROOT = r"d:\Buet_Datathon"
MODEL_ID = "bengaliAI/tugstugi_bengaliai-regional-asr_whisper-medium"
CHUNK_CSV = os.path.join(PROJECT_ROOT, "data", "processed", "train_final.csv")
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "models", "whisper-bengali-lora")
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_DTYPE = torch.float16 if device == "cuda" else torch.float32
MAX_LABEL_LEN = 448


# 1. LOAD DATASET

print("Loading dataset...")
if not os.path.exists(CHUNK_CSV):
    raise FileNotFoundError(f"Missing CSV: {CHUNK_CSV}")

df = pd.read_csv(CHUNK_CSV)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

train_limit = int(len(df) * 0.9)
train_df = df.iloc[:train_limit]
eval_df = df.iloc[train_limit:]

train_dataset = Dataset.from_pandas(train_df)
eval_dataset = Dataset.from_pandas(eval_df)

print(f"Train samples: {len(train_dataset)}")
print(f"Eval samples: {len(eval_dataset)}")


# 2. LOAD BASE MODEL

print("Loading base model...")
processor = WhisperProcessor.from_pretrained(MODEL_ID)

model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=MODEL_DTYPE,
    low_cpu_mem_usage=True,
).to(device)
model.config.use_cache = False
model.to(device)


# 3. APPLY LORA

print("Applying LoRA...")
lora_config = LoraConfig(
    r=32,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


# 4. PREPROCESSING

def prepare_dataset(batch):
    try:
        audio_path = batch["filename"]
        wav, sr = torchaudio.load(audio_path)

        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        wav = wav.squeeze(0)

        if sr != 16000:
            wav = torchaudio.functional.resample(wav, sr, 16000)

        batch["input_features"] = processor.feature_extractor(
            wav.numpy(),
            sampling_rate=16000
        ).input_features[0]

        batch["labels"] = processor.tokenizer(
            batch["transcript"],
            truncation=True,
            max_length=MAX_LABEL_LEN
        ).input_ids

    except Exception as e:
        print(f"Error processing {batch.get('filename', 'unknown')}: {e}")
        batch["input_features"] = None
        batch["labels"] = None

    return batch

print("Processing data...")
train_dataset = train_dataset.map(prepare_dataset)
eval_dataset = eval_dataset.map(prepare_dataset)

# Important fix: filter valid rows only
train_dataset = train_dataset.filter(lambda x: x["input_features"] is not None and x["labels"] is not None)
eval_dataset = eval_dataset.filter(lambda x: x["input_features"] is not None and x["labels"] is not None)

print(f"Train usable: {len(train_dataset)}")
print(f"Eval usable : {len(eval_dataset)}")


# 5. DATA COLLATOR

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    feature_dtype: torch.dtype = torch.float16

    def __call__(self, features):
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # force same dtype as model
        batch["input_features"] = batch["input_features"].to(self.feature_dtype)

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(
    processor=processor,
    feature_dtype=MODEL_DTYPE
)


# 6. WER METRICS

wer_metric = evaluate.load("wer")

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer}


# 7. TRAINING ARGUMENTS LoRA-friendly

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=1e-4,
    warmup_steps=50,
    max_steps=3000,
    gradient_checkpointing=True,
    fp16=(device == "cuda"),
    fp16_full_eval=(device == "cuda"),
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    logging_steps=25,
    report_to=["tensorboard"],
    predict_with_generate=True,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
)
trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor.feature_extractor,
)


# 8. START TRAINING

print("Starting LoRA Training...")
trainer.train()

model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Training Complete! Model saved to {OUTPUT_DIR}")


Loading dataset...
Train samples: 11250
Eval samples: 1250
Loading base model...


Loading weights: 100%|██████████| 947/947 [00:00<00:00, 967.53it/s, Materializing param=model.encoder.layers.23.self_attn_layer_norm.weight]    


Applying LoRA...
trainable params: 9,437,184 || all params: 773,295,104 || trainable%: 1.2204
Processing data...


Filter: 100%|██████████| 1250/1250 [01:10<00:00, 17.62 examples/s]


Train usable: 11250
Eval usable : 1250
Starting LoRA Training...


Step,Training Loss,Validation Loss,Wer
500,1.479201,0.382959,36.207617
1000,1.379077,0.357865,34.753334
1500,1.290115,0.348728,32.784830
2000,1.241599,0.342403,32.984091
2500,1.212829,0.341019,33.403503
3000,1.164485,0.340172,32.487546


Training Complete! Model saved to d:\Buet_Datathon\models\whisper-bengali-lora
